In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# RecursiveCharacterTextSplitter is a text splitter that splits text based on character count, while trying to split at natural break points (like newlines or punctuation) when possible. It is useful for splitting long documents into smaller chunks for processing.
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

## Vector Stores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

/home/valac/Projects/RAG/RAG-Udemi/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sorces
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query into embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduce Hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge

### 1. Sample Data

In [3]:
## Create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. It allows systems to learn from data, identify patterns, and make decisions with minimal human intervention. Machine learning is widely used in various applications, including image recognition, natural language processing, and predictive analytics.
    """,
    """
    Deep Learning and Neural Networks
    
    Deep Learning is a subset of machine learning that involves neural networks with many layers. It is particularly effective for tasks such as image and speech recognition, natural language processing, and autonomous driving. Deep learning models require large amounts of data and computational power but can achieve state-of-the-art performance in various domains.
    """,
    """
    Natural Language Processing (NLP)
    
    Natural Language Processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and humans through natural language. It involves the development of algorithms and models that enable machines to understand, interpret, and generate human language. NLP is used in various applications, including sentiment analysis, machine translation, and chatbots.
    """
]

print(sample_docs)

['\n    Machine Learning Fundamentals\n\n    Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. It allows systems to learn from data, identify patterns, and make decisions with minimal human intervention. Machine learning is widely used in various applications, including image recognition, natural language processing, and predictive analytics.\n    ', '\n    Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning that involves neural networks with many layers. It is particularly effective for tasks such as image and speech recognition, natural language processing, and autonomous driving. Deep learning models require large amounts of data and computational power but can achieve state-of-the-art performance in various domains.\n    ', '\n    Natural Language Processing (NLP)\n\n    Natural Language Processing (NLP) 

In [4]:
## Save sample documents to files
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt", "w") as f:
        f.write(doc)

print(f"Documents saved to {temp_dir}")

Documents saved to /tmp/tmpgdcbhj26


In [5]:
## Save sample documents to files
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"./data/doc_{i}.txt", "w") as f:
        f.write(doc)

print(f"Documents saved to ./data")

Documents saved to ./data


### 2. Document Loading

In [6]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documents = loader.load()

print(f"Loaded {len(documents)} documents.")
print(documents[0].page_content[:500])  # Print the first 500 characters of the first document


Loaded 3 documents.

    Deep Learning and Neural Networks

    Deep Learning is a subset of machine learning that involves neural networks with many layers. It is particularly effective for tasks such as image and speech recognition, natural language processing, and autonomous driving. Deep learning models require large amounts of data and computational power but can achieve state-of-the-art performance in various domains.
    


### 3. Document Splitting

In [7]:
## initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,  # Maximum number of characters in each chunk
    chunk_overlap = 30,  # Number of characters to overlap between chunks
    length_function = len,  # Function to calculate the length of the text (using character count in this case)
    ## separators=["\n\n", "\n", ". ", " ", ""] # Hierarchy of separators to split the text (try to split at double newlines first, then single newlines, then periods, then spaces, and finally if no other separator is found, split at the character limit)
    separators=[" "]
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents.")
print(f"\nChunk Example:\n")
print(f"Content: {chunks[0].page_content[:100]}")  # Print the first 500 characters of the first chunks
print(f"Metadata: {chunks[0].metadata}")

Created 6 chunks from 3 documents.

Chunk Example:

Content: Deep Learning and Neural Networks

    Deep Learning is a subset of machine learning that involves n
Metadata: {'source': 'data/doc_1.txt'}


In [8]:
chunks

[Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning that involves neural networks with many layers. It is particularly effective for tasks such as image and speech recognition, natural language processing, and autonomous driving. Deep learning models require'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep learning models require large amounts of data and computational power but can achieve state-of-the-art performance in various domains.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. It allows systems to learn from data, identify patterns, and make'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='identif

###  3. Embedding Models

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [10]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
)

sample_text = "Machine learning is a fantastic field of study"
vector_query = embedding.embed_query(sample_text)
vector_query


[-0.037994384765625,
 -0.00629425048828125,
 -0.02569580078125,
 0.0180816650390625,
 0.047149658203125,
 -0.030059814453125,
 0.006175994873046875,
 0.06365966796875,
 -0.0184326171875,
 0.03253173828125,
 0.0120086669921875,
 -0.015380859375,
 -0.04058837890625,
 -0.0247650146484375,
 0.00750732421875,
 0.0268096923828125,
 -0.0296783447265625,
 -0.030029296875,
 0.068603515625,
 0.04742431640625,
 0.0235443115234375,
 0.01099395751953125,
 0.02288818359375,
 -0.043670654296875,
 0.010986328125,
 -0.033172607421875,
 -0.005275726318359375,
 0.044097900390625,
 0.01486968994140625,
 0.01031494140625,
 0.0142974853515625,
 0.0106353759765625,
 -0.0582275390625,
 -0.0232391357421875,
 0.0031604766845703125,
 0.04736328125,
 0.028228759765625,
 0.0002315044403076172,
 -0.0599365234375,
 0.02789306640625,
 -0.00885009765625,
 0.0013265609741210938,
 0.028656005859375,
 0.06427001953125,
 -0.0206756591796875,
 -0.02850341796875,
 -0.0273590087890625,
 -0.0259857177734375,
 0.058349609375,


### 3.1 Initialize ChromaDB Vector Store and store chunks in vector representation

In [11]:
## Create a chromadb vector store
persist_directory = "./chroma_db"

## Initialize chromadb with openAI embeddings
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print(f"Vector store created with {len(vector_store)} documents.")
print(f"Vector store persisted at: {persist_directory}")
print(f"Collection name: {vector_store._collection.name}")
print(f"Vector store created with: {vector_store._collection.count()} documents.")

Vector store created with 6 documents.
Vector store persisted at: ./chroma_db
Collection name: rag_collection
Vector store created with: 6 documents.


### 4. Similarity Search

In [12]:
## test the similarity search

query = "What are the types of machine learning?"

similar_docs = vector_store.similarity_search(query, k=3)
similar_docs

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. It allows systems to learn from data, identify patterns, and make'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='identify patterns, and make decisions with minimal human intervention. Machine learning is widely used in various applications, including image recognition, natural language processing, and predictive analytics.'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning that involves neural networks with many layers. It is particularly effective for tasks such as image and speech recognition, natural language processing, and autonomous driving. Deep learning models require')

In [13]:
query = "What is NLP?"

similar_docs = vector_store.similarity_search(query, k=3)
similar_docs

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='machines to understand, interpret, and generate human language. NLP is used in various applications, including sentiment analysis, machine translation, and chatbots.'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    Natural Language Processing (NLP) is a field of artificial intelligence that focuses on the interaction between computers and humans through natural language. It involves the development of algorithms and models that enable machines to understand,'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning that involves neural networks with many layers. It is particularly effective for tasks such as image and speech recognition, natural language processing, and autonomous driving. Deep learning models require')]

In [14]:
query = "What is Deep Learning?"

similar_docs = vector_store.similarity_search(query, k=3)
similar_docs

[Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning that involves neural networks with many layers. It is particularly effective for tasks such as image and speech recognition, natural language processing, and autonomous driving. Deep learning models require'),
 Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep learning models require large amounts of data and computational power but can achieve state-of-the-art performance in various domains.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. It allows systems to learn from data, identify patterns, and make')]

### 41. Advance similarity search with score

In [15]:
results_scores = vector_store.similarity_search_with_score(query, k=3)
print(results_scores)

[(Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning that involves neural networks with many layers. It is particularly effective for tasks such as image and speech recognition, natural language processing, and autonomous driving. Deep learning models require'), 0.6020141839981079), (Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep learning models require large amounts of data and computational power but can achieve state-of-the-art performance in various domains.'), 0.9052531719207764), (Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. It allows systems to learn from data, identify patterns, and make'), 1.1616123914718628)]


#### 4.1.1 Understanding Similarity Scores

The similarity score represents how closely related a document chunk si to your query. The score depends on the distance metric used:

ChromaDB default metric: L2 distance(euclidean distance)
- Lower scores: MORE similar (closer in vector space)
- Score 0: identical vectors
- Typical range: 0 to 2 (but can be higher)

Cosine similarity(If configured):
-Higher scores: MORE similar
- Range: -1 to 1 (1 being identical)

### Initialize LLM, RAG Chain, Prompt Template, Query the RAG system

In [16]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    ## low temperature for more deterministic responses. this means that the model will be more likely to choose the most probable next word, resulting in more focused and consistent answers.
    ## higher temperature would make the model more creative and diverse in its responses, but it may also lead to less coherent and more random answers.
    temperature=0.2,
    max_tokens=500,
)

In [17]:
test_response = llm.invoke("What is Large Language Models?")
test_response

AIMessage(content="Large Language Models (LLMs) are a type of artificial intelligence model that are trained on vast amounts of text data to understand and generate human language. These models are capable of performing a wide range of natural language processing tasks, such as text generation, translation, summarization, and question answering. LLMs have significantly advanced the field of natural language processing and have been used in various applications, including chatbots, language translation, and content generation. Some popular examples of LLMs include OpenAI's GPT (Generative Pre-trained Transformer) models and Google's BERT (Bidirectional Encoder Representations from Transformers) model.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 13, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_t

In [18]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model("openai:gpt-3.5-turbo")
llm.invoke("What is AI?")

AIMessage(content='AI, or artificial intelligence, refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. It involves the development of algorithms and technologies that enable computers and machines to perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI applications can be found in a wide range of industries, including healthcare, finance, transportation, and entertainment.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 11, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DITqBsMM9yAzqs

### Modern RAG Chain

In [19]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

#### Convert vector store to retriever

In [20]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3} ## Retrieve the top 3 most similar documents from the vector store based on the query
)

retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7898aa14acf0>, search_kwargs={'k': 3})

#### Create Prompt template


In [22]:
system_prompt = """
You are a helpful assistant for answering questions about machine learning, deep learning, and natural language processing. Use the following retrieved context to provide accurate and concise answers to the user's questions. If you don't know the answer, say you don't know.
Use three sentences maximum and keep the answer concise and to the point. Do not include any information that is not relevant to the question.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant for answering questions about machine learning, deep learning, and natural language processing. Use the following retrieved context to provide accurate and concise answers to the user's questions. If you don't know the answer, say you don't know.\nUse three sentences maximum and keep the answer concise and to the point. Do not include any information that is not relevant to the question.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

#### Create Document Chain
this ill combine all the relevante context and push it inside the prompt

In [23]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant for answering questions about machine learning, deep learning, and natural language processing. Use the following retrieved context to provide accurate and concise answers to the user's questions. If you don't know the answer, say you don't know.\nUse three sentences maximum and keep the answer concise and to the point. Do not include any information that is not relevant to the question.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, te

#### Create the Final RAG Chain

- retriever will fetch relevant information
- document_chain will process de documents with the LLM

In [26]:
rag_chain = create_retrieval_chain(retriever, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7898aa14acf0>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant for answering questions about machine learning, deep learning, and natural language processing. Use the 

#### Execute The RAG Chain
give the input parameter for the place holder

In [29]:
response = rag_chain.invoke({"input":"What is Deep Learning?"})
response

{'input': 'What is Deep Learning?',
 'context': [Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep Learning is a subset of machine learning that involves neural networks with many layers. It is particularly effective for tasks such as image and speech recognition, natural language processing, and autonomous driving. Deep learning models require'),
  Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep learning models require large amounts of data and computational power but can achieve state-of-the-art performance in various domains.'),
  Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit instructions. It allows systems to learn from data, identify patterns, and make')],
 'answer': 'Deep Le

In [30]:
response["answer"]

'Deep Learning is a subset of machine learning that uses neural networks with many layers to learn patterns from data. It is effective for tasks like image recognition, speech processing, and natural language understanding. Deep learning models require large datasets and computational power to achieve high-performance results.'

In [31]:
# function to query the modern RAG system
def query_rag_modern(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Using create_retrieval_chain approach
    result =  rag_chain.invoke({"input": question})
    print(f"Answer: {result['answer']}")
    print("\nRetrieved Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

    return result

# Test Queries
test_questions = [
    "What are three types of Machine Learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs bes used for?"
]

for question in test_questions:
    query_rag_modern(question)
    print("\n" + "="*80 + "\n")


Question: What are three types of Machine Learning?
--------------------------------------------------
Answer: Three types of Machine Learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning involves training a model on labeled data, unsupervised learning deals with unlabeled data to find patterns, and reinforcement learning uses a system of rewards and punishments to guide the model's learning process.

Retrieved Context:

--- Source 1 ---
Machine Learning Fundamentals

    Machine Learning is a subset of artificial intelligence that focuses on the development of algorithms and statistical models that enable computers to perform tasks w...

--- Source 2 ---
identify patterns, and make decisions with minimal human intervention. Machine learning is widely used in various applications, including image recognition, natural language processing, and predictive...

--- Source 3 ---
Deep Learning and Neural Networks

    Deep Learning is a subse